In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

# 1. Silence TensorFlow C++ logs (0=all, 1=no info, 2=no warnings, 3=no errors)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# 2. Specifically silence the "absl" and CUDA initialization warnings
os.environ['TF_CPP_MAX_VLOG_LEVEL'] = '0'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import warnings
warnings.filterwarnings('ignore')

# NOW you can import tensorflow
import tensorflow as tf

# 3. Silence Python-level TensorFlow logger
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ==========================================
# 1. SETTINGS & PATHS
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings('ignore')

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

2026-04-20 03:33:02.167078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776655982.576011      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776655982.696696      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776655983.814575      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776655983.814621      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776655983.814624      55 computation_placer.cc:177] computation placer alr

In [3]:
# Paths from your environment
TRAIN_DIR = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train"
TEST_DIR = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final"

# ==========================================
# 2. DATA LOADERS
# ==========================================
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_ds.class_names # e.g., ['cats', 'dogs']
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

Found 4733 files belonging to 2 classes.
Using 3787 files for training.


I0000 00:00:1776656028.060971      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776656028.067020      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 4733 files belonging to 2 classes.
Using 946 files for validation.


In [4]:
# ==========================================
# 3. ARCHITECTURE (ConvNeXt-Tiny + Deep Head)
# ==========================================
def build_convnext_model():
    augmentation = models.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.3),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomTranslation(0.1, 0.1)
    ], name="augmentation")

    base_model = tf.keras.applications.ConvNeXtTiny(
        weights='imagenet', 
        include_top=False, 
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False # Frozen for Stage 1

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    
    # ConvNeXt feature extraction
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    
    # --- Deep Regularized Head ---
    x = layers.Dense(512, activation='gelu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(256, activation='gelu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(128, activation='gelu')(x)
    x = layers.GaussianNoise(0.1)(x)
    
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    return models.Model(inputs, outputs), base_model

model, base_model = build_convnext_model()

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [5]:
# ==========================================
# 4. STAGE 1: TRAIN HEAD
# ==========================================
print("\n--- Stage 1: Training Deep Head ---")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)


--- Stage 1: Training Deep Head ---


In [ ]:
history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=15, 
    callbacks=callbacks
)

Epoch 1/15


I0000 00:00:1776656073.614585     145 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776656075.091999     145 service.cc:152] XLA service 0x7cacf604c980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776656075.092036     145 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776656075.092040     145 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776656075.577149     145 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-04-20 03:34:38.899525: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-20 03:34:39.040306: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured t

118/119 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step - accuracy: 0.8962 - loss: 0.3933

2026-04-20 03:35:19.190735: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-20 03:35:19.325370: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - accuracy: 0.8966 - loss: 0.3927

2026-04-20 03:35:36.320212: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-20 03:35:36.455129: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


119/119 ━━━━━━━━━━━━━━━━━━━━ 77s 483ms/step - accuracy: 0.8970 - loss: 0.3921 - val_accuracy: 0.9937 - val_loss: 0.2572 - learning_rate: 0.0010
Epoch 2/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 50s 418ms/step - accuracy: 0.9763 - loss: 0.2670 - val_accuracy: 0.9958 - val_loss: 0.2190 - learning_rate: 0.0010
Epoch 3/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 54s 454ms/step - accuracy: 0.9808 - loss: 0.2493 - val_accuracy: 0.9958 - val_loss: 0.2162 - learning_rate: 0.0010
Epoch 4/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 443ms/step - accuracy: 0.9877 - loss: 0.2359 - val_accuracy: 0.9968 - val_loss: 0.2117 - learning_rate: 0.0010
Epoch 5/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 446ms/step - accuracy: 0.9889 - loss: 0.2275 - val_accuracy: 0.9979 - val_loss: 0.2080 - learning_rate: 0.0010
Epoch 6/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 447ms/step - accuracy: 0.9913 - loss: 0.2267 - val_accuracy: 0.9979 - val_loss: 0.2108 - learning_rate: 0.0010
Epoch 7/15
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 444ms/step - accuracy: 0.9913 - loss:

In [ ]:
# ==========================================
# 5. STAGE 2: FINE-TUNING
# ==========================================
print("\n--- Stage 2: Fine-Tuning ConvNeXt ---")
base_model.trainable = True

# Use a very low learning rate for fine-tuning
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

In [ ]:
history_fine = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=10, 
    callbacks=callbacks
)

In [ ]:
# ==========================================
# 6. VISUALIZATION
# ==========================================
def plot_results(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']
    
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Train Acc', color='teal')
    plt.plot(val_acc, label='Val Acc', color='coral')
    plt.axvline(x=len(h1.history['accuracy'])-1, color='red', ls='--')
    plt.title('ConvNeXt Accuracy'); plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Train Loss', color='teal')
    plt.plot(val_loss, label='Val Loss', color='coral')
    plt.axvline(x=len(h1.history['loss'])-1, color='red', ls='--')
    plt.title('ConvNeXt Loss'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()

plot_results(history, history_fine)

# ==========================================
# 7. PREDICTION & SUBMISSION
# ==========================================
print("\n--- Generating Submission ---")

# Get list of test files
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

def process_test_image(filename):
    path = os.path.join(TEST_DIR, filename)
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img

# Create test dataset
test_ds = tf.data.Dataset.from_tensor_slices(test_files).map(process_test_image).batch(BATCH_SIZE)

# Predict
probabilities = model.predict(test_ds).flatten()
# Map 0/1 back to class names (cats/dogs)
predictions = [class_names[int(p >= 0.5)] for p in probabilities]

# Create and save CSV
submission = pd.DataFrame({
    'ID': test_files,
    'Label': predictions
})

submission.to_csv("submission.csv", index=False)
print("Done! 'submission.csv' is saved and ready for upload.")